In [1]:
import urllib.parse
import urllib.request
import xml.etree.ElementTree as ET
import time

BASE_URL = "https://export.arxiv.org/api/query"

params = {
    "search_query": "all:computer",
    "start": 0,
    "max_results": 5
}

# Build query URL safely
query_url = BASE_URL + "?" + urllib.parse.urlencode(params)

request = urllib.request.Request(
    query_url,
    headers={
        "User-Agent": "arxiv-data-project/1.0 (research script)"
    }
)

time.sleep(3)

# Make request
with urllib.request.urlopen(request) as response:
    xml_data = response.read()

# Parse XML
root = ET.fromstring(xml_data)

# arXiv uses Atom namespace
ns = {"atom": "http://www.w3.org/2005/Atom"}

papers = []

for entry in root.findall("atom:entry", ns):
    title = entry.find("atom:title", ns).text.strip()
    summary = entry.find("atom:summary", ns).text.strip()
    published = entry.find("atom:published", ns).text
    
    authors = [
        author.find("atom:name", ns).text
        for author in entry.findall("atom:author", ns)
    ]

    pdf_link = next((link.attrib.get("href") for link in entry.findall("atom:link", ns) 
                if link.attrib.get("title") == "pdf" and link.attrib.get("type") == "application/pdf" and link.attrib.get("rel") == "related"), 
                None
            )

    papers.append({
        "title": title,
        "authors": authors,
        "published": published,
        "summary": summary,
        "pdf_link": pdf_link
    })

# Display parsed results
for paper in papers:
    print("\nTITLE:", paper["title"])
    print("AUTHORS:", ", ".join(paper["authors"]))
    print("PUBLISHED:", paper["published"])
    print("SUMMARY:", paper["summary"][:200], "...")
    print("PDF LINK:", paper["pdf_link"])


TITLE: Vision Based Game Development Using Human Computer Interaction
AUTHORS: S. Sumathi, S. K. Srivatsa, M. Uma Maheswari
PUBLISHED: 2010-02-10T19:46:07Z
SUMMARY: A Human Computer Interface (HCI) System for playing games is designed here for more natural communication with the machines. The system presented here is a vision-based system for detection of long vo ...
PDF LINK: https://arxiv.org/pdf/1002.2191v1

TITLE: From truth to computability I
AUTHORS: Giorgi Japaridze
PUBLISHED: 2004-07-21T03:58:22Z
SUMMARY: The recently initiated approach called computability logic is a formal theory of interactive computation. See a comprehensive online source on the subject at http://www.cis.upenn.edu/~giorgi/cl.html . ...
PDF LINK: https://arxiv.org/pdf/cs/0407054v2

TITLE: Changing Neighbors k Secure Sum Protocol for Secure Multi Party Computation
AUTHORS: Rashid Sheikh, Beerendra Kumar, Durgesh Kumar Mishra
PUBLISHED: 2010-02-11T19:58:10Z
SUMMARY: Secure sum computation of private data inpu

### EXTRACT PDF TEXT

In [17]:
# trying pymupdf4llm
import pymupdf4llm
import urllib.request
import os

pdf_dir = "pdfs"
os.makedirs(pdf_dir, exist_ok = True)

pdf_path = "https://arxiv.org/pdf/1002.2191v1"
local_pdf = os.path.join(pdf_dir, "Vision Based Game Development Using HCI.pdf")
urllib.request.urlretrieve(pdf_path, local_pdf)

pages = pymupdf4llm.to_markdown(local_pdf, page_chunks=True)

print(f"Pages extracted: {len(pages)}")
print(pages[0]["metadata"])
print(pages[0]["text"][:1000])


OCR disabled because Tesseract language data not found.
Pages extracted: 7
{'format': 'PDF 1.4', 'title': 'Microsoft Word - 07011060 IJCSIS Camera Ready Paper.doc', 'author': 'RAZVI', 'subject': '', 'keywords': '', 'creator': 'PScript5.dll Version 5.2.2', 'producer': 'Acrobat Distiller 8.1.0 (Windows)', 'creationDate': "D:20100202231332+05'00'", 'modDate': "D:20100208131620+05'00'", 'trapped': '', 'encryption': None, 'file_path': 'pdfs\\Vision Based Game Development Using HCI.pdf', 'page_count': 7, 'page_number': 1}
_(IJCSIS) International Journal of Computer Science and Information Security, Vol. 7, No. 1, 2010_ 

## Vision Based Game Development Using Human Computer Interaction 

Bharath  University                   St.Joseph College of Engineering                   Bharath University Chennai,India                                       Chennai,India                                     Chennai,India 

_**Abstract**_ **— A Human Computer Interface (HCI) System for playing games is des

In [25]:
for page in pages:
    if len(page["text"].strip()) <50:
         print(f"Short/empty page: {page['metadata']}")

#checking the character count distribution
lengths = [len(p["text"]) for p in pages]
print(f'Average chars per page: {sum(lengths)/len(lengths):.0f}')
print(f"Min: {min(lengths)}, Max: {max(lengths)}")

Average chars per page: 3487
Min: 2493, Max: 4217


### CLEAN PDF

In [55]:
# cleaning extracted text
import re

def clean_arxiv_page_text(text: str) -> str:
    if not text:
        return ""
    
    # Join words split by hyphen + line break: "learn-\ning" -> "learning"
    text = re.sub(r"([A-Za-z])-\s*\n\s*([A-Za-z])", r"\1\2", text)

    # Join broken lines inside paragraphs: "This is a line\nthat continues." -> "This is a line that continues."
    text = re.sub(r"(?<!\n)\n(?!\n)", " ", text)

    # Remove extra spaced again after line joins
    text = re.sub(r"[ ]{2,}", " ", text)

    # Replace multiple line breaks with a single space
    text = re.sub(r"\n+", " ", text)

    # Replace multiple spaces with a single space
    text = re.sub(r"\s+", " ", text)

    # Remove common arXiv/DOI/vol/no/version boilerplate
    text = re.sub(r"arXiv:\s*\S+", " ", text, flags=re.IGNORECASE)
    text = re.sub(r"\bdoi:\s*\S+", " ", text, flags=re.IGNORECASE)
    text = re.sub(r"\bVol\.?\s*\d+\b", " ", text, flags=re.I)  # remove volume markers
    text = re.sub(r"\bNo\.?\s*\d+\b", " ", text, flags=re.I)  
    text = re.sub(r"\bv\d+\b", " ", text, flags=re.IGNORECASE)  # version tokens like v1, v2

    # Remove page markers
    text = re.sub(r"^\s*page\s*\d+(\s*of\s*\d+)?\s*$", " ", text, flags=re.IGNORECASE | re.MULTILINE)
    text = re.sub(r"^\s*\d+\s*$", " ", text, flags=re.MULTILINE)  # lone page number lines
    
    # Remove repeated "downloaded from..." style footer lines
    text = re.sub(r"^\s*downloaded\s+from\s+.*$", " ", text, flags=re.IGNORECASE | re.MULTILINE)

    #Remove markdown artifacts like "## " or "- " at the start of lines
    text = re.sub(r"(?m)^#{1,6}\s*", " ", text)

    # Normalize line endings
    text = text.replace("\r\n", "\n").replace("\r", "\n")

    #Normalize whitespace again 
    text = re.sub(r"\s{2,}", " ", text).strip()

    #Remove excessive blank lines
    text = re.sub(r"\n{3,}", "\n\n", text)

    # Remove strange repeated punctuation only lines like ### or ***
    text = re.sub(r"(?m)^[#*=\-_.~`]{2,}$", " ", text)

    #Fix spaces before punctuation
    text = re.sub(r"\s+([.,;:!?])", r"\1", text)

    return text.strip()

In [56]:
clean_pages = []
for page in pages:
    cleaned = clean_arxiv_page_text(page["text"])
    clean_pages.append({
        "text": cleaned,
        "metadata": page["metadata"]
    })

print("Pages before:", len(pages))
print("Pages after cleaning:", len(clean_pages))
print("Sample cleaned page text:", clean_pages[0]["text"][:1000] if clean_pages else "N/A")

Pages before: 7
Pages after cleaning: 7
Sample cleaned page text: _(IJCSIS) International Journal of Computer Science and Information Security,,, 2010_ ## Vision Based Game Development Using Human Computer Interaction Bharath University St.Joseph College of Engineering Bharath University Chennai,India Chennai,India Chennai,India _**Abstract**_ **— A Human Computer Interface (HCI) System for playing games is designed here for more natural communication with the machines. The system presented here is a vision-based system for detection of long voluntary eye blinks and interpretation of blink patterns for communication between man and machine. This system replaces the mouse with the human face as a new way to interact with the computer. Facial features (nose tip and eyes) are detected and tracked in real-time to use their actions as mouse events. The coordinates and movement of the nose tip in the live video feed are translated to become the coordinates and movement of the mouse pointer o

In [57]:
# filtering low quality chunks

def is_low_quality_chunk(text: str, min_chars=100, min_alpha=25, min_words=10, max_symbol_ratio=0.35, max_citation_density=0.12):
    if not text:
        return True
    
    stripped = text.strip()
    
    if len(stripped) < min_chars:
        return True
    
    words = stripped.split()
    if len(words) < min_words:
        return True
    
    total_chars = len(stripped)

    if total_chars == 0:
        return True
 
    alpha_chars = sum(c.isalpha() for c in stripped)
    alpha_ratio = alpha_chars / total_chars
    if alpha_ratio < 0.5:
        return True
    
    symbol_chars = sum(not ch.isalnum() and not ch.isspace() for ch in stripped)
    if symbol_chars / total_chars > max_symbol_ratio:
        return True
    
    cites = re.findall(r"\[\d+(?:,\s*\d+)*\]", stripped)
    citation_density = len(cites) / max(len(words), 1)
    if citation_density > max_citation_density:
        return True
    
    return False

In [58]:
#chunking
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=200,
    separators=["\n\n", "\n", ". ", " ", ""]
)

clean_chunks = []

for page in clean_pages:
    splits = text_splitter.split_text(page["text"])

    for s in splits:
        if not is_low_quality_chunk(s):
            clean_chunks.append({
                "text": s,
                "metadata": page["metadata"]
            })

print("Clean chunks kept:", len(clean_chunks))
print("Sample clean chunk:", clean_chunks[0]["text"][:350] if clean_chunks else "N/A")

Clean chunks kept: 77
Sample clean chunk: _(IJCSIS) International Journal of Computer Science and Information Security,,, 2010_ ## Vision Based Game Development Using Human Computer Interaction Bharath University St.Joseph College of Engineering Bharath University Chennai,India Chennai,India Chennai,India _**Abstract**_ **— A Human Computer Interface (HCI) System for playing games is desig


In [ ]:
# #chunking
# from langchain_text_splitters import RecursiveCharacterTextSplitter

# text_splitter = RecursiveCharacterTextSplitter(
#     chunk_size=500,
#     chunk_overlap=200,
#     separators=["\n\n", "\n", ". ", " ", ""]
# )

# chunks = []

# for page in pages:
#     splits = text_splitter.split_text(page["text"])

#     for s in splits:
#         chunks.append({
#             "chunk_id": f"{page['metadata']['page_number']}_{len(chunks)}",
#             "source": local_pdf,
#             "text": s,
#             "metadata": page["metadata"],
#             "page_number": page["metadata"]["page_number"]
#         })

# print(len(chunks))
# print(chunks[0])


80
{'chunk_id': '1_0', 'source': 'pdfs\\Vision Based Game Development Using HCI.pdf', 'text': '_(IJCSIS) International Journal of Computer Science and Information Security, Vol. 7, No. 1, 2010_ \n\n## Vision Based Game Development Using Human Computer Interaction \n\nBharath  University                   St.Joseph College of Engineering                   Bharath University Chennai,India                                       Chennai,India                                     Chennai,India', 'metadata': {'format': 'PDF 1.4', 'title': 'Microsoft Word - 07011060 IJCSIS Camera Ready Paper.doc', 'author': 'RAZVI', 'subject': '', 'keywords': '', 'creator': 'PScript5.dll Version 5.2.2', 'producer': 'Acrobat Distiller 8.1.0 (Windows)', 'creationDate': "D:20100202231332+05'00'", 'modDate': "D:20100208131620+05'00'", 'trapped': '', 'encryption': None, 'file_path': 'pdfs\\Vision Based Game Development Using HCI.pdf', 'page_count': 7, 'page_number': 1}, 'page_number': 1}


In [59]:
import spacy

nlp = spacy.load("en_core_web_md")

In [60]:
docs = nlp.pipe((c["text"] for c in clean_chunks), batch_size=32)

for i, doc in enumerate(docs):
    ents = [(ent.text, ent.label_) for ent in doc.ents[:10]]
    if i < 3:
        print("Chunk", i, "entities:", ents)

Chunk 0 entities: [('International Journal of Computer Science and Information Security', 'ORG'), ('2010', 'DATE'), ('## Vision', 'MONEY'), ('Engineering Bharath University Chennai', 'ORG'), ('India Chennai', 'GPE'), ('India Chennai', 'GPE'), ('India', 'GPE'), ('Human Computer Interface', 'ORG')]
Chunk 1 entities: []
Chunk 2 entities: []


In [ ]:
# testing tokenization on chunks from the pdf file
for chunk in chunks:
    print(f"Chunk metadata: {chunk['metadata']}")
    doc = nlp(chunk["text"])
    for token in doc:
        print(token.text)


Chunk metadata: {'format': 'PDF 1.4', 'title': 'Microsoft Word - 07011060 IJCSIS Camera Ready Paper.doc', 'author': 'RAZVI', 'subject': '', 'keywords': '', 'creator': 'PScript5.dll Version 5.2.2', 'producer': 'Acrobat Distiller 8.1.0 (Windows)', 'creationDate': "D:20100202231332+05'00'", 'modDate': "D:20100208131620+05'00'", 'trapped': '', 'encryption': None, 'file_path': 'pdfs\\Vision Based Game Development Using HCI.pdf', 'page_count': 7, 'page_number': 1}
_
(
IJCSIS
)
International
Journal
of
Computer
Science
and
Information
Security
,
Vol
.
7
,
No
.
1
,
2010
_



#
#
Vision
Based
Game
Development
Using
Human
Computer
Interaction



Bharath
 
University
                  
St.
Joseph
College
of
Engineering
                  
Bharath
University
Chennai
,
India
                                      
Chennai
,
India
                                    
Chennai
,
India
Chunk metadata: {'format': 'PDF 1.4', 'title': 'Microsoft Word - 07011060 IJCSIS Camera Ready Paper.doc', 'author': 'RAZ